# Fine-tune the retrieval model on financial QA

Retrieval is the measured weak link in this pipeline. On `data/samples/small_filing.txt` the average similarity of retrieved chunks was **0.593** — below the 0.6 routing bar — so the re-query loop fires, spends its whole retry budget, and generates from weak evidence anyway. Reformulation pulled it to 0.643, which is a patch on the symptom, not the cause.

This notebook fine-tunes `BAAI/bge-large-en-v1.5` on financial question/passage pairs and measures whether that moves the numbers, using the evaluation the repo already has.

**What makes this worth doing rather than another `from_pretrained` call:** every number here is about your own system. The baseline is measured, the change is trained, and the result is measured again with the same harness. If it does not improve, that is the finding and it goes in the README.

---

### Plan

1. Baseline the stock model on **FiQA test** (1,706 held-out pairs, 57k-passage corpus) — a real retrieval benchmark with a published reference
2. Fine-tune on **FiQA train** (14,166 pairs) with `MultipleNegativesRankingLoss`
3. Re-measure on the same held-out split
4. Re-measure **through the repo's own pipeline** on the FinanceBench corpus, which is a transfer setting: trained on FiQA, evaluated on filings

### Before you run

- **Runtime → Change runtime type → T4 GPU.** A T4 is enough: step 7 uses GradCache to train at an effective batch of 64 within 16GB, so you get the same number of in-batch negatives a bigger card would give, just slower. Pick L4 or A100 instead if your Colab Pro account offers them and you want it faster.
- **Not the TPU.** `sentence-transformers` is PyTorch, so a v5e would mean PyTorch/XLA, where variable-length text batches trigger constant recompilation. Debugging that would cost more time than the entire fine-tune. TPUs are for large-scale JAX training, not a one-hour encoder fine-tune.
- **Push your local commits first.** This clones from GitHub; step 3 checks the clone is current.

## 1. GPU

In [ ]:
!nvidia-smi

import torch

assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> GPU"
name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"\n{name}  |  {vram:.0f} GB VRAM")

## 2. Clone and install

In [ ]:
REPO = "https://github.com/abhinaba01/fundamental-financial-analysis.git"

import os

if not os.path.exists("fundamental-financial-analysis"):
    !git clone -q {REPO}

%cd fundamental-financial-analysis
!git log --oneline -3

In [ ]:
!pip install -q -e ".[eval]"
!pip install -q "sentence-transformers>=3.0"
!python -m spacy download en_core_web_sm -q

import sentence_transformers
print("sentence-transformers", sentence_transformers.__version__)

## 3. Check the clone is current

Step 8 evaluates through the repo's own scripts, which need `--embedding-model` flags added alongside this notebook. Without them the pipeline-level comparison would silently score the stock model twice.

In [ ]:
from pathlib import Path

checks = {
    "--embedding-model (scripts/index_eval_corpus.py)":
        "--embedding-model" in Path("scripts/index_eval_corpus.py").read_text(),
    "--embedding-model (evaluation/eval_rag.py)":
        "--embedding-model" in Path("evaluation/eval_rag.py").read_text(),
    "prepare_eval_datasets.py":
        Path("scripts/prepare_eval_datasets.py").exists(),
}
for label, ok in checks.items():
    print(f"  {'OK     ' if ok else 'MISSING'}  {label}")

if not all(checks.values()):
    raise SystemExit(
        "\nThis clone predates the flags this notebook needs.\n"
        "Push your local commits, delete the folder, and re-run step 2:\n"
        "    git push origin main\n"
    )
print("\nClone is current.")

## 4. Load FiQA

FiQA-2018 is the standard financial retrieval benchmark: 57,638 passages, 6,648 questions, and relevance judgements split train/validation/test. Training on `train` and evaluating on `test` is the conventional setup — and unlike the Financial PhraseBank situation elsewhere in this repo, the evaluation split is genuinely held out.

In [ ]:
from datasets import load_dataset

corpus_ds = load_dataset("BeIR/fiqa", "corpus", split="corpus")
queries_ds = load_dataset("BeIR/fiqa", "queries", split="queries")
qrels = load_dataset("BeIR/fiqa-qrels")

corpus = {row["_id"]: (row["title"] + " " + row["text"]).strip() for row in corpus_ds}
queries = {row["_id"]: row["text"] for row in queries_ds}

print(f"corpus  {len(corpus):,}")
print(f"queries {len(queries):,}")
print({split: len(rows) for split, rows in qrels.items()})

In [ ]:
from collections import defaultdict


def relevant_map(split):
    """query_id -> set of relevant corpus_ids, keeping only positive judgements."""
    mapping = defaultdict(set)
    for row in qrels[split]:
        if row["score"] > 0:
            mapping[str(row["query-id"])].add(str(row["corpus-id"]))
    return dict(mapping)


train_rel = relevant_map("train")
test_rel = relevant_map("test")
print(f"train queries {len(train_rel):,}   test queries {len(test_rel):,}")

### Build the evaluator

`InformationRetrievalEvaluator` encodes the whole corpus and ranks it per query, so it reports NDCG@10 / Recall@k / MRR the way MTEB does. Published reference for the stock model on FiQA2018 is **NDCG@10 ≈ 0.45**.

Scoring against all 57k passages is slow. `EVAL_CORPUS_EXTRA` shrinks it to the relevant passages plus a random sample of distractors — smaller haystack, easier task, so the absolute number will read higher than MTEB's. Set it to `None` for a directly comparable figure, at the cost of a much longer encode.

In [ ]:
import random

from sentence_transformers.evaluation import InformationRetrievalEvaluator

EVAL_CORPUS_EXTRA = 20_000   # None -> score against the full 57k corpus
random.seed(42)

eval_queries = {qid: queries[qid] for qid in test_rel if qid in queries}
needed = {cid for cids in test_rel.values() for cid in cids}

if EVAL_CORPUS_EXTRA:
    distractors = [cid for cid in corpus if cid not in needed]
    sampled = random.sample(distractors, min(EVAL_CORPUS_EXTRA, len(distractors)))
    eval_corpus = {cid: corpus[cid] for cid in list(needed) + sampled}
else:
    eval_corpus = corpus

print(f"evaluating {len(eval_queries):,} queries against {len(eval_corpus):,} passages")

ir_evaluator = InformationRetrievalEvaluator(
    queries=eval_queries,
    corpus=eval_corpus,
    relevant_docs={qid: test_rel[qid] for qid in eval_queries},
    name="fiqa-test",
    show_progress_bar=True,
    corpus_chunk_size=10_000,
)

## 5. Baseline the stock model

Measure before changing anything. Without this number the fine-tune proves nothing.

In [ ]:
from sentence_transformers import SentenceTransformer

BASE_MODEL = "BAAI/bge-large-en-v1.5"   # the model src/preprocessing/embedder.py uses

base_model = SentenceTransformer(BASE_MODEL, device="cuda")
baseline = ir_evaluator(base_model)

baseline_ndcg = baseline["fiqa-test_cosine_ndcg@10"]
print(f"\nBASELINE NDCG@10: {baseline_ndcg:.4f}")
for key in sorted(baseline):
    if any(k in key for k in ("ndcg@10", "recall@5", "recall@10", "mrr@10", "precision@1")):
        print(f"  {key.replace('fiqa-test_cosine_', ''):<14} {baseline[key]:.4f}")

## 6. Training pairs

`MultipleNegativesRankingLoss` takes (anchor, positive) pairs and treats every other positive in the batch as a negative. That has one consequence worth understanding: **batch size is the number of negatives**, so it drives quality more than epochs do. Bigger batch, harder contrastive task, better model.

It also means a batch must never contain two rows sharing a correct answer — a true positive sitting in the negatives teaches the model the opposite of the intended lesson. `no_duplicates` batching enforces that.

In [ ]:
from datasets import Dataset

pairs = [
    {"anchor": queries[qid], "positive": corpus[cid]}
    for qid, cids in train_rel.items()
    if qid in queries
    for cid in cids
    if cid in corpus
]

train_dataset = Dataset.from_list(pairs).shuffle(seed=42)
print(f"{len(train_dataset):,} training pairs")
print(train_dataset[0])

## 7. Fine-tune

Batch size **is** the negative count for this loss, so it drives quality more than epochs or learning rate. The obvious problem on a 16GB T4 is that a 335M-parameter encoder at batch 64 does not fit.

GradCache solves exactly this. `CachedMultipleNegativesRankingLoss` runs the batch through in `mini_batch_size` pieces and caches the gradients, so memory tracks the mini-batch while the loss still sees every in-batch negative. A T4 can therefore train at an effective batch of 64 — the same contrastive signal an A100 would give — at the cost of extra forward passes.

| GPU | VRAM | Batch | Loss |
|-----|------|-------|------|
| A100 | 40GB | 64 | plain MNRL |
| L4 | 24GB | 32 | plain MNRL |
| **T4** | **16GB** | **64 (mini 8)** | **CachedMNRL — GradCache** |

The cell picks from detected VRAM. On a T4 expect roughly 2x the wall clock of the plain loss for the same batch; that is the trade GradCache makes, and it is worth it here because the alternative is 16 negatives instead of 64.

One epoch is the norm for this loss — more usually overfits without improving retrieval.

In [ ]:
from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.losses import (
    CachedMultipleNegativesRankingLoss,
    MultipleNegativesRankingLoss,
)
from sentence_transformers.training_args import BatchSamplers

# Batch size is the negative count, so keep it high. GradCache lets a small
# card do that: memory tracks mini_batch_size while the loss still sees the
# whole batch. On a large card the plain loss is the same signal without the
# extra forward passes.
if vram > 35:
    BATCH, MINI = 64, None
elif vram > 20:
    BATCH, MINI = 32, None
else:
    BATCH, MINI = 64, 8

OUTPUT_DIR = "bge-large-fiqa"

model = SentenceTransformer(BASE_MODEL, device="cuda")

if MINI:
    loss = CachedMultipleNegativesRankingLoss(model, mini_batch_size=MINI)
    print(f"GradCache: effective batch {BATCH}, {MINI} in memory at a time ({vram:.0f}GB)")
else:
    loss = MultipleNegativesRankingLoss(model)
    print(f"standard MNRL: batch {BATCH} ({vram:.0f}GB)")

args = SentenceTransformerTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=BATCH,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    fp16=True,
    # A batch holding two rows with the same correct answer would put a true
    # positive into the in-batch negatives and train against it.
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    logging_steps=50,
    save_strategy="no",
    report_to="none",
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    loss=loss,
)
trainer.train()

model.save_pretrained(OUTPUT_DIR)
print(f"\nsaved to {OUTPUT_DIR}")

## 8. Re-measure on the held-out split

In [ ]:
tuned = ir_evaluator(model)
tuned_ndcg = tuned["fiqa-test_cosine_ndcg@10"]

print(f"{'metric':<16}{'baseline':>10}{'tuned':>10}{'delta':>10}")
for key in sorted(baseline):
    if any(k in key for k in ("ndcg@10", "recall@5", "recall@10", "mrr@10", "precision@1")):
        short = key.replace("fiqa-test_cosine_", "")
        before, after = baseline[key], tuned[key]
        print(f"{short:<16}{before:>10.4f}{after:>10.4f}{after - before:>+10.4f}")

change = (tuned_ndcg - baseline_ndcg) / baseline_ndcg * 100
print(f"\nNDCG@10: {baseline_ndcg:.4f} -> {tuned_ndcg:.4f}  ({change:+.1f}%)")
if tuned_ndcg <= baseline_ndcg:
    print("\nNo improvement. That is a result, not a failed run - report it.")
    print("Most likely causes, in order: batch too small (few in-batch")
    print("negatives), learning rate too high, or the base model already")
    print("saturating this benchmark.")

## 9. Does it help the actual pipeline?

FiQA is forum-style financial Q&A; your pipeline retrieves from SEC filing prose. A gain on FiQA test does not automatically transfer, so this measures the model through the repo's own retrieval evaluation on the FinanceBench corpus — trained on FiQA, evaluated on filings.

Retrieval metrics only, so no `OPENAI_API_KEY` and no API spend. Generation falls back to extractive synthesis; ignore the EM/ROUGE fields and read Hit@5 and MRR.

In [ ]:
!python scripts/prepare_eval_datasets.py --dataset financebench

In [ ]:
# Each model needs its own collection: querying vectors written by a different
# model compares two unrelated embedding spaces and returns nonsense silently.
!python scripts/index_eval_corpus.py --gpu \
  --collection fb_base --vector-store data/eval_vs_base

!python scripts/index_eval_corpus.py --gpu \
  --collection fb_tuned --vector-store data/eval_vs_tuned \
  --embedding-model {OUTPUT_DIR}

In [ ]:
import json

!python -m evaluation.eval_rag --test-set data/eval/financebench_test.json --run-agent \
  --collection fb_base --vector-store data/eval_vs_base \
  --output data/eval/fb_base.json

!python -m evaluation.eval_rag --test-set data/eval/financebench_test.json --run-agent \
  --collection fb_tuned --vector-store data/eval_vs_tuned \
  --embedding-model {OUTPUT_DIR} \
  --output data/eval/fb_tuned.json

before = json.load(open("data/eval/fb_base.json"))
after = json.load(open("data/eval/fb_tuned.json"))

print(f"\n{'metric':<16}{'base':>10}{'tuned':>10}")
for key in ("hit_at_1", "hit_at_5", "mrr", "ndcg"):
    if key in before and key in after:
        print(f"{key:<16}{before[key]:>10.4f}{after[key]:>10.4f}")

print("\nFull payloads:")
print(" base :", {k: v for k, v in before.items() if isinstance(v, (int, float))})
print(" tuned:", {k: v for k, v in after.items() if isinstance(v, (int, float))})

## 10. Publish and wire in

Only worth doing if step 8 or 9 actually improved. If neither did, the honest move is to write the negative result into the README and leave the stock model in place.

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
HF_REPO = "YOUR_HF_USERNAME/bge-large-fiqa-financial"   # <- edit

model.push_to_hub(HF_REPO)
print(f"https://huggingface.co/{HF_REPO}")
print("\nTo use it in the pipeline, set MODEL_NAME in")
print("src/preprocessing/embedder.py to that id, and delete")
print("data/vector_store/ so the corpus re-embeds in the new space.")

## Notes

- **A regression is a publishable result.** Say what you measured and what you think caused it. That reads better than a project with no negative results in it, which usually means none were looked for.
- **Re-embed after switching models.** Vectors from two models are not comparable, and ChromaDB will not warn you — it returns confident nonsense. Delete `data/vector_store/` when changing `MODEL_NAME`.
- **Absolute numbers here are not MTEB numbers** unless you set `EVAL_CORPUS_EXTRA = None`. The sampled corpus is a smaller haystack and scores higher. The before/after *delta* is valid either way, since both are measured on the same corpus.
- **Colab Pro background execution** keeps this running if you close the tab; the free tier will not.